# T3.2 Analizar e visualizar estatísticas de datos baixados de MongoDB
## Conexión a MongoDB dende Python e análise de datos

En Atlas mete o dataset de exemplo na túa base de datos.

Engade o enderezo IP autorizado a conectar para que poidas conectar á base de datos.

Proba a conectar dende compass coa URL de conexión que che da.

Da BBDD de proba: sample_mfix, na colección movies: https://www.mongodb.com/docs/atlas/sample-data/sample-mflix/

Conecta a Atlas dende Python, carga a colección en Pandas. Averigua o modo máis adecuado.

In [2]:
from pymongo import MongoClient

#HOST="(...).mongodb.net"
HOST="cluster0.nl8aj.mongodb.net"
PORT=27017
USERNAME="madalinismana365"
PASSWORD="Madalin13245678..."

if HOST == 'localhost':
    if not USERNAME:
        cli_mongo = MongoClient(HOST, PORT)
    else:
        cli_mongo = MongoClient(HOST, PORT, USERNAME, PASSWORD)
else:
    cli_mongo = MongoClient(f"mongodb+srv://{USERNAME}:{PASSWORD}@{HOST}/test")

print (cli_mongo.list_database_names())
##usar la base de sample_mflix
db_mflix=cli_mongo['sample_mflix']


['Estudantes2', 'PruebaClase', 'RickyMorty', 'covid', 'sample_mflix', 'sample_restaurants', 'sample_supplies', 'admin', 'local']


A) Contar o total de películas.


In [3]:
total_movies = db_mflix.movies.count_documents({})
print(f"\nA) Total de películas: {total_movies}")


A) Total de películas: 21349


B) Contar o número de películas de cada xénero.

In [4]:
pipeline_genero = [
    {"$unwind": "$genres"},
    {"$group": {"_id": "$genres", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
result_genero = list(db_mflix.movies.aggregate(pipeline_genero))
print("\nB) Películas por género:")
for genre in result_genero[:10]:  # Mostrar solo los 10 primeros
    print(f"{genre['_id']}: {genre['count']}")


B) Películas por género:
Drama: 12385
Comedy: 6532
Romance: 3318
Crime: 2457
Thriller: 2454
Action: 2381
Adventure: 1900
Documentary: 1834
Horror: 1470
Biography: 1269


C) Contar cantas películas hai por ano.

In [5]:
pipeline_ano = [
    {"$match": {"year": {"$exists": True}}},
    {"$group": {"_id": "$year", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]
result_ano = list(db_mflix.movies.aggregate(pipeline_ano))
print("\nC) Películas por año (mostrando solo algunos años):")
for year in result_ano[::50]:  # Mostrar cada 50 años para no saturar
    print(f"{year['_id']}: {year['count']}")


C) Películas por año (mostrando solo algunos años):
1896: 2
1959: 71
2009: 917


D) Mirar se hai correlación entre imdb e rotten tomatoes.

In [6]:
pipeline_ratings = [
    {"$match": {
        "imdb.rating": {"$exists": True, "$ne": ""},
        "tomatoes.viewer.rating": {"$exists": True, "$ne": ""}
    }},
    {"$project": {
        "imdb_rating": {"$toDouble": "$imdb.rating"},
        "tomatoes_rating": "$tomatoes.viewer.rating",
        "title": 1
    }},
    {"$limit": 10}  # Limitar para el ejemplo
]
result_ratings = list(db_mflix.movies.aggregate(pipeline_ratings))
print("\nD) Ejemplo de ratings de películas (IMDB vs Rotten Tomatoes):")
for movie in result_ratings:
    print(f"{movie['title']}: IMDB {movie['imdb_rating']}, Tomatoes {movie['tomatoes_rating']}")


D) Ejemplo de ratings de películas (IMDB vs Rotten Tomatoes):
Winsor McCay, the Famous Cartoonist of the N.Y. Herald and His Moving Comics: IMDB 7.3, Tomatoes 3.4
From Hand to Mouth: IMDB 7.0, Tomatoes 3.3
Miss Lulu Bett: IMDB 7.2, Tomatoes 2.5
Tol'able David: IMDB 8.1, Tomatoes 3.3
The Great Train Robbery: IMDB 7.4, Tomatoes 3.7
Gertie the Dinosaur: IMDB 7.3, Tomatoes 3.7
Civilization: IMDB 6.3, Tomatoes 0.0
The Poor Little Rich Girl: IMDB 6.9, Tomatoes 3.9
The Blue Bird: IMDB 6.6, Tomatoes 3.6
The Four Horsemen of the Apocalypse: IMDB 7.9, Tomatoes 3.9


E) Contar cantas películas ten cada director

In [7]:
pipeline_directores = [
    {"$unwind": "$directors"},
    {"$group": {"_id": "$directors", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
result_directores = list(db_mflix.movies.aggregate(pipeline_directores))
print("\nE) Top 10 directores con más películas:")
for director in result_directores:
    print(f"{director['_id']}: {director['count']}")


E) Top 10 directores con más películas:
Woody Allen: 40
Martin Scorsese: 32
Takashi Miike: 31
John Ford: 29
Sidney Lumet: 29
Steven Spielberg: 29
Robert Altman: 27
Spike Lee: 27
Clint Eastwood: 27
Michael Apted: 27


F) Contar en cantas películas ten participado cada actor

In [8]:
pipeline_actores = [
    {"$unwind": "$cast"},
    {"$group": {"_id": "$cast", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
result_actores = list(db_mflix.movies.aggregate(pipeline_actores))
print("\nF) Top 10 actores con más participaciones:")
for actor in result_actores:
    print(f"{actor['_id']}: {actor['count']}")


F) Top 10 actores con más participaciones:
Gèrard Depardieu: 67
Robert De Niro: 58
Michael Caine: 51
Bruce Willis: 49
Morgan Freeman: 48
Samuel L. Jackson: 48
Christopher Plummer: 47
Gene Hackman: 46
Max von Sydow: 45
Nicolas Cage: 45


G) Contar cantas películas hai de cada idioma

In [9]:
pipeline_idioma = [
    {"$unwind": "$languages"},
    {"$group": {"_id": "$languages", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
result_idioma = list(db_mflix.movies.aggregate(pipeline_idioma))
print("\nG) Top 10 idiomas con más películas:")
for lang in result_idioma:
    print(f"{lang['_id']}: {lang['count']}")



G) Top 10 idiomas con más películas:
English: 14639
French: 2405
Spanish: 1600
German: 1377
Italian: 1228
Japanese: 860
Russian: 845
Mandarin: 478
Hindi: 464
Portuguese: 346


H) Cal é a media de premios das películas

In [10]:
pipeline_premios = [
    {"$match": {"awards.wins": {"$exists": True}}},
    {"$group": {
        "_id": None,
        "avg_wins": {"$avg": "$awards.wins"},
        "avg_nominations": {"$avg": "$awards.nominations"}
    }}
]
result_premios = list(db_mflix.movies.aggregate(pipeline_premios))
print("\nH) Media de premios:")
print(f"Victorias promedio: {result_premios[0]['avg_wins']:.2f}")
print(f"Nominaciones promedio: {result_premios[0]['avg_nominations']:.2f}")


H) Media de premios:
Victorias promedio: 4.06
Nominaciones promedio: 4.83


J) Amosar as películas que teñen alomenos 3 premios

In [11]:
pipeline_min_premios = [
    {"$match": {"awards.wins": {"$gte": 3}}},
    {"$project": {"title": 1, "awards.wins": 1}},
    {"$sort": {"awards.wins": -1}},
    {"$limit": 10}
]
result_min_premios = list(db_mflix.movies.aggregate(pipeline_min_premios))
print("\nJ) Películas con al menos 3 premios (top 10):")
for movie in result_min_premios:
    print(f"{movie['title']}: {movie['awards']['wins']} premios")


J) Películas con al menos 3 premios (top 10):
12 Years a Slave: 267 premios
Gravity: 231 premios
Gravity: 231 premios
Birdman: Or (The Unexpected Virtue of Ignorance): 210 premios
Boyhood: 185 premios
The Lord of the Rings: The Return of the King: 175 premios
No Country for Old Men: 172 premios
The Social Network: 171 premios
Inception: 162 premios
The Artist: 161 premios


K) Contar as películas por país

In [12]:
pipeline_pais = [
    {"$unwind": "$countries"},
    {"$group": {"_id": "$countries", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
result_pais = list(db_mflix.movies.aggregate(pipeline_pais))
print("\nK) Top 10 países con más películas:")
for country in result_pais:
    print(f"{country['_id']}: {country['count']}")


K) Top 10 países con más películas:
USA: 10921
UK: 2652
France: 2647
Germany: 1494
Canada: 1260
Italy: 1217
Japan: 786
Spain: 675
India: 564
Australia: 470
